# Plain asynchronous FL (notebook 02)

The simplest async method: every update is applied **the moment it arrives**, and nobody waits.

**How it works**

1. A client downloads the current global model and trains.
2. When its update arrives: `global += server_rate × (client model − model it started from)`.
3. The client immediately starts again on the newest global model.

> **Caution:** slow clients deliver updates computed on **old** global models (high staleness). With no correction, these stale updates can pull the model backwards, so accuracy oscillates a lot.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FedAsync(Approach):
    name = "fedasync"
    label = "Async no buffer"          # shown in plot legends
    server_rate = 0.5             # how far the global model moves toward each update

    def on_update(self, server, update):
        new_model = apply_delta(server.model, [update], [1.0], self.server_rate)
        server.set_model(new_model, [update])

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedAsync(server_rate=0.2))`. It is saved as a separate run.

In [3]:
results = run(FedAsync())

fedasync                     seed=42   val= 61.96%  acc= 64.20%  time=   560.2s  updates=  100  worst10%= 51.91%  wall=6.2s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
47,42,560.1644,100,0.642,1.7462,0.5191,8.0
